# DeepEval을 활용한 파인튜닝 모델 평가


## 학습 목표
1. DeepEval의 다양한 평가지표 이해
2. 파인튜닝 모델 평가를 위한 적절한 지표 선택
3. Base Model vs Fine-tuned Model 성능 비교
4. 평가 결과 시각화 및 분석


## 데이터셋 정보
- **도메인**: 소아 영양 상담
- **데이터**: nutrition.csv (200개 Q&A 쌍)
- **Base Model**: LGAI-EXAONE/EXAONE-3.5-2.4B-Instruct
- **Fine-tuned Model**: good593/EXAONE-3.5-2.4B-fine-tuning


## 1. DeepEval 평가지표 소개

### DeepEval이란?
- LLM 애플리케이션을 위한 종합 평가 프레임워크
- 다양한 평가 메트릭을 제공하여 모델의 품질을 정량적으로 측정
- LLM을 Judge로 사용하여 평가 (GPT-4, Claude 등)

### 주요 평가지표

#### 답변 품질 평가
- **Answer Relevancy (답변 관련성)**
   - 질문과 답변이 얼마나 관련있는지 측정
   - 질문의 핵심 의도를 파악하여 답변이 적절한지 평가

- **Faithfulness (충실성)**
   - 답변이 참고 자료(reference)에 얼마나 충실한지 측정
   - 환각(hallucination) 감지에 효과적
   - 참고 답변에 없는 내용을 추가하거나 왜곡하지 않았는지 확인

- **Hallucination (환각)**
   - 모델이 사실이 아닌 정보를 생성했는지 검출
   - Context나 Reference와 모순되는 내용 식별
   

#### RAG 시스템 평가
- **Contextual Relevancy (맥락 관련성)**
   - 검색된 컨텍스트가 질문과 얼마나 관련있는지

- **Contextual Precision (맥락 정밀도)**
   - 관련 있는 컨텍스트가 상위에 랭킹되었는지

- **Contextual Recall (맥락 재현율)**
   - 필요한 모든 정보가 컨텍스트에 포함되었는지
   

#### 안전성 평가
- **Bias (편향)**
   - 모델 답변에 편향된 내용이 있는지 검사

- **Toxicity (유해성)**
   - 유해하거나 공격적인 내용이 있는지 검사

#### 커스텀 평가
- **G-Eval**
   - 사용자 정의 평가 기준 설정 가능
   - 도메인 특화 평가 지표 생성


## 2. 파인튜닝 모델 평가를 위한 지표 선택

### 케이스 분석
- **시나리오**: 소아 영양 상담 Q&A
- **목표**: 전문적이고 정확한 답변 제공
- **참고 데이터**: 전문가가 작성한 답변 (reference)


### 선택한 평가지표와 이유

#### Answer Relevancy (답변 관련성)
**선택 이유:**
- 질문에 대한 답변이 적절한지 확인
- 파인튜닝 후 질문 이해도가 향상되었는지 측정
- 예: "아기 이유식 언제 시작?"이라는 질문에 이유식 시작 시기를 명확히 답하는지

**예제:**
```
질문: "아기 비타민D 꼭 먹여야 하나요?"
좋은 답변 (high relevancy): "비타민 D는 아이에게 매우 중요하며, 돌까지 하루 400 IU, 그 이후는 600 IU 권장합니다."
나쁜 답변 (low relevancy): "비타민은 다양한 종류가 있으며 영양소가 중요합니다."
```

#### Faithfulness (충실성)
**선택 이유:**
- 참고 답변의 내용을 정확히 반영하는지 확인
- 잘못된 정보나 환각 방지
- 의료/영양 도메인에서 매우 중요 (잘못된 정보는 위험)

**예제:**
```
참고 답변: "두 돌까지는 음식에 소금을 따로 첨가하지 않는 것이 좋습니다."
충실한 답변: "두 돌까지는 소금을 따로 넣지 않고 원래 짠 음식도 피하세요."
불충실한 답변: "한 돌부터는 소금을 조금씩 넣어도 됩니다." (잘못된 정보)
```

#### G-Eval (도메인 특화 평가)
**선택 이유:**
- 소아 영양 상담 특화 기준 평가
- 전문성, 안전성, 이해도를 종합 평가
- 커스텀 기준: "전문 용어의 적절한 설명", "안전 주의사항 포함 여부"

**평가 기준 예시:**
1. 전문성: 의학적으로 정확한 정보 제공
2. 안전성: 주의사항이나 의사 상담 권장 포함
3. 명확성: 부모가 이해하기 쉬운 설명
4. 실용성: 구체적인 행동 지침 제공

## 3. 환경 설정


In [ ]:
import sys
import tqdm

tqdm.tqdm = tqdm.std.tqdm
if hasattr(tqdm, 'notebook'):
    tqdm.notebook.tqdm = tqdm.std.tqdm
    tqdm.auto.tqdm = tqdm.std.tqdm

In [2]:
from dotenv import load_dotenv

load_dotenv()

True

## 4. 데이터 로드

In [3]:
import pandas as pd

# 데이터 로드
df = pd.read_csv('data/nutrition.csv')

print(f"데이터셋 크기: {len(df)}개")
print(f"컬럼: {df.columns.tolist()}")

데이터셋 크기: 198개
컬럼: ['user_input', 'reference', 'reference_video_url', 'persona_name', 'query_style', 'query_length']


> 데이터 샘플

In [4]:
print(f"질문: {df['user_input'].iloc[0]}")
print(f"참고 답변:\n{df['reference'].iloc[0]}")

질문: 체중에 대한 잘못된 정보는 무엇인가요?
참고 답변:
'하루 1,000ml만 넘지 않으면 괜찮다'는 이야기는 잘못된 정보입니다. 실제로 하루 1,000ml 정도 먹는 아기는 드물고, 이런 아기들 중에는 체중 과다나 이유식 문제 등 다른 문제가 있는 경우가 많습니다.


> 평가용 샘플 선택 (전체 데이터는 시간이 오래 걸리므로 일부만 사용)

In [5]:
# 평가용 샘플 선택 (전체 데이터는 시간이 오래 걸리므로 일부만 사용)
sample_size = 20  # 실습용으로 20개만 사용
df_sample = df.sample(n=sample_size, random_state=42).reset_index(drop=True)
print(f"평가용 샘플 {sample_size}개 추출 완료")


평가용 샘플 20개 추출 완료


In [6]:
df_sample.head(2)

,user_input,reference,reference_video_url,persona_name,query_style,query_length
0,까망베르 치즈를 어린아이에게 주면 안 되는 이유는 무엇인가요?,까망베르와 같은 연성 치즈는 멸균 처리되지 않은 경우가 많아 면역력이 약한 어린아이...,https://www.youtube.com/watch?v=OB2vXyWDriQ,a pediatric nutrition expert,MISSPELLED,MEDIUM
1,보리차로 분유를 탈 수 있나요? 왜 안되나요?,분유는 반드시 맹물(끓인 물)로 타야 하며 보리차나 다른 차로 타서는 안 됩니다.,https://www.youtube.com/watch?v=sLDedsQBmDU,a pediatric nutrition expert,POOR_GRAMMAR,LONG


## 5. 모델 로드 및 답변 생성


In [7]:
import torch

# 디바이스 설정
device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"사용 디바이스: {device}")


사용 디바이스: cuda


### Base 모델 로드

In [8]:
from transformers import AutoTokenizer, AutoModelForCausalLM

# Base 모델 로드
base_model_name = "LGAI-EXAONE/EXAONE-3.5-2.4B-Instruct"

base_tokenizer = AutoTokenizer.from_pretrained(base_model_name)
base_model = AutoModelForCausalLM.from_pretrained(
    base_model_name,
    device_map="auto",
    trust_remote_code=True
)

print("Base 모델 로드 완료")


tokenizer_config.json: 0.00B [00:00, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/563 [00:00<?, ?B/s]

...

model-00002-of-00002.safetensors:   0%|          | 0.00/4.65G [00:00<?, ?B/s]

model-00001-of-00002.safetensors:   0%|          | 0.00/4.98G [00:00<?, ?B/s]

Loading checkpoint shards:   0%|          | 0/2 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/134 [00:00<?, ?B/s]

Base 모델 로드 완료


### Fine-tuned 모델 로드

In [ ]:
# Fine-tuned 모델 로드
finetuned_model_name = "good593/EXAONE-3.5-2.4B-fine-tuning"

finetuned_tokenizer = AutoTokenizer.from_pretrained(finetuned_model_name)
finetuned_model = AutoModelForCausalLM.from_pretrained(
    finetuned_model_name,
    device_map="auto",
    trust_remote_code=True
)

print("Fine-tuned 모델 로드 완료")


### 답변 생성 함수

In [ ]:
# 답변 생성 함수
def generate_answer(question, model, tokenizer, max_length=512):
    """모델을 사용하여 질문에 대한 답변 생성"""
    # 프롬프트 구성
    prompt = f"질문: {question}\n\n답변:"
    
    # 토큰화
    inputs = tokenizer(prompt, return_tensors="pt", truncation=True, max_length=512)
    inputs = {k: v.to(model.device) for k, v in inputs.items()}
    
    # 답변 생성
    with torch.no_grad():
        outputs = model.generate(
            **inputs,
            max_new_tokens=max_length,
            temperature=0.7,
            do_sample=True,
            top_p=0.9,
            pad_token_id=tokenizer.eos_token_id
        )
    
    # 디코딩
    full_text = tokenizer.decode(outputs[0], skip_special_tokens=True)
    
    # 답변 부분만 추출
    if "답변:" in full_text:
        answer = full_text.split("답변:")[-1].strip()
    else:
        answer = full_text.replace(prompt, "").strip()
    
    return answer

print("답변 생성 함수 준비 완료")


### 각 모델의 답변 생성

In [ ]:
from tqdm import tqdm

# 각 모델의 답변 생성
print("Base 모델 답변 생성 중...")
df_sample['base_answer'] = [
    generate_answer(question, base_model, base_tokenizer) 
    for question in tqdm(df_sample['user_input'], desc="Base Model")
]

print("\nFine-tuned 모델 답변 생성 중...")
df_sample['finetuned_answer'] = [
    generate_answer(question, finetuned_model, finetuned_tokenizer) 
    for question in tqdm(df_sample['user_input'], desc="Fine-tuned Model")
]

print("\n모든 답변 생성 완료!")


> 샘플 결과 확인

In [ ]:
# 샘플 결과 확인

sample_idx = 0
print(f"\n질문:\n{df_sample['user_input'].iloc[sample_idx]}")
print(f"\n참고 답변:\n{df_sample['reference'].iloc[sample_idx]}")
print(f"\nBase 모델 답변:\n{df_sample['base_answer'].iloc[sample_idx]}")
print(f"\nFine-tuned 모델 답변:\n{df_sample['finetuned_answer'].iloc[sample_idx]}")


## 6. DeepEval 평가 메트릭 적용


### 평가 메트릭 초기화

In [ ]:
from deepeval.metrics import AnswerRelevancyMetric

# 1. Answer Relevancy Metric
answer_relevancy = AnswerRelevancyMetric(
    threshold=0.7,
    model="gpt-4",
    include_reason=True
)

In [ ]:
from deepeval.metrics import FaithfulnessMetric

# 2. Faithfulness Metric
faithfulness = FaithfulnessMetric(
    threshold=0.7,
    model="gpt-4",
    include_reason=True
)

In [ ]:
from deepeval.metrics import GEval
from deepeval.test_case import LLMTestCase, LLMTestCaseParams

# 3. G-Eval (커스텀 평가)
geval = GEval(
    name="Pediatric Nutrition Expertise",
    criteria="의료적 정확성, 안전성 주의사항 포함, 명확한 설명, 실용적 조언을 평가합니다.",
    evaluation_steps=[
        "답변이 의학적으로 정확한 정보를 포함하는지 확인",
        "안전 주의사항이나 의사 상담 권장이 적절히 포함되었는지 확인",
        "부모가 이해하기 쉬운 명확한 설명인지 확인",
        "구체적이고 실행 가능한 조언을 제공하는지 확인"
    ],
    evaluation_params=[
        LLMTestCaseParams.INPUT,
        LLMTestCaseParams.ACTUAL_OUTPUT,
        LLMTestCaseParams.EXPECTED_OUTPUT
    ],
    threshold=0.7,
    model="gpt-4"
)


### 평가

> 평가 함수

In [ ]:
import numpy as np

# 평가 함수
def evaluate_model(df, answer_column, model_name):
    """모델 답변에 대해 모든 메트릭 평가"""
    results = {
        'relevancy_scores': [],
        'faithfulness_scores': [],
        'geval_scores': []
    }
    
    print(f"\n{'='*80}")
    print(f"{model_name} 평가 중...")
    print(f"{'='*80}\n")
    
    for idx, row in tqdm(df.iterrows(), total=len(df), desc=f"Evaluating {model_name}"):
        # Test Case 생성
        test_case = LLMTestCase(
            input=row['user_input'],
            actual_output=row[answer_column],
            expected_output=row['reference']
        )
        
        try:
            # Answer Relevancy 평가
            answer_relevancy.measure(test_case)
            results['relevancy_scores'].append(answer_relevancy.score)
            
            # Faithfulness 평가
            faithfulness.measure(test_case)
            results['faithfulness_scores'].append(faithfulness.score)
            
            # G-Eval 평가
            geval.measure(test_case)
            results['geval_scores'].append(geval.score)
            
        except Exception as e:
            print(f"평가 중 오류 발생 (행 {idx}): {e}")
            results['relevancy_scores'].append(0)
            results['faithfulness_scores'].append(0)
            results['geval_scores'].append(0)
    
    # 평균 점수 계산
    avg_scores = {
        'answer_relevancy': np.mean(results['relevancy_scores']),
        'faithfulness': np.mean(results['faithfulness_scores']),
        'geval': np.mean(results['geval_scores'])
    }
    
    print(f"\n{model_name} 평가 완료")
    print(f"   - Answer Relevancy: {avg_scores['answer_relevancy']:.3f}")
    print(f"   - Faithfulness: {avg_scores['faithfulness']:.3f}")
    print(f"   - G-Eval: {avg_scores['geval']:.3f}")
    
    return results, avg_scores


> Base 모델 평가

In [ ]:
# Base 모델 평가
base_results, base_avg = evaluate_model(df_sample, 'base_answer', 'Base Model')

> Fine-tuned 모델 평가

In [ ]:
# Fine-tuned 모델 평가
finetuned_results, finetuned_avg = evaluate_model(df_sample, 'finetuned_answer', 'Fine-tuned Model')

> 결과를 DataFrame으로 정리

In [ ]:
# 결과를 DataFrame으로 정리
results_df = pd.DataFrame({
    'Metric': ['Answer Relevancy', 'Faithfulness', 'G-Eval'],
    'Base Model': [base_avg['answer_relevancy'], base_avg['faithfulness'], base_avg['geval']],
    'Fine-tuned Model': [finetuned_avg['answer_relevancy'], finetuned_avg['faithfulness'], finetuned_avg['geval']]
})

results_df['Improvement'] = ((results_df['Fine-tuned Model'] - results_df['Base Model']) / results_df['Base Model'] * 100).round(2)


In [ ]:
print("\n" + "="*80)
print("평가 결과 요약")
print("="*80)
print(results_df.to_string(index=False))


## 7. 평가 결과 시각화 및 분석


### 그래프 1: 평균 점수 비교 막대 그래프

In [ ]:
import matplotlib.pyplot as plt
import koreanize_matplotlib

# 그래프 1: 평균 점수 비교 막대 그래프
fig, ax = plt.subplots(1, 1, figsize=(12, 6))

x = np.arange(len(results_df['Metric']))
width = 0.35

bars1 = ax.bar(x - width/2, results_df['Base Model'], width, label='Base Model', color='#FF6B6B', alpha=0.8)
bars2 = ax.bar(x + width/2, results_df['Fine-tuned Model'], width, label='Fine-tuned Model', color='#4ECDC4', alpha=0.8)

ax.set_xlabel('평가 지표', fontsize=12, fontweight='bold')
ax.set_ylabel('점수', fontsize=12, fontweight='bold')
ax.set_title('Base Model vs Fine-tuned Model: 평가 지표 비교', fontsize=14, fontweight='bold', pad=20)
ax.set_xticks(x)
ax.set_xticklabels(results_df['Metric'], fontsize=11)
ax.legend(fontsize=11)
ax.set_ylim(0, 1.0)
ax.grid(axis='y', alpha=0.3, linestyle='--')

# 막대 위에 점수 표시
for bars in [bars1, bars2]:
    for bar in bars:
        height = bar.get_height()
        ax.text(bar.get_x() + bar.get_width()/2., height,
                f'{height:.3f}',
                ha='center', va='bottom', fontsize=10, fontweight='bold')

plt.tight_layout()
plt.show()



### 그래프 2: 성능 향상률 시각화

In [ ]:
# 그래프 2: 성능 향상률 시각화
fig, ax = plt.subplots(1, 1, figsize=(12, 6))

colors = ['#2ECC71' if x > 0 else '#E74C3C' for x in results_df['Improvement']]
bars = ax.barh(results_df['Metric'], results_df['Improvement'], color=colors, alpha=0.8)

ax.set_xlabel('성능 향상률 (%)', fontsize=12, fontweight='bold')
ax.set_ylabel('평가 지표', fontsize=12, fontweight='bold')
ax.set_title('Fine-tuning 후 성능 향상률', fontsize=14, fontweight='bold', pad=20)
ax.axvline(x=0, color='black', linestyle='-', linewidth=0.8)
ax.grid(axis='x', alpha=0.3, linestyle='--')

# 막대 끝에 향상률 표시
for i, (bar, improvement) in enumerate(zip(bars, results_df['Improvement'])):
    width = bar.get_width()
    ax.text(width + (2 if width > 0 else -2), bar.get_y() + bar.get_height()/2.,
            f'{improvement:+.2f}%',
            ha='left' if width > 0 else 'right', va='center', 
            fontsize=11, fontweight='bold')

plt.tight_layout()
plt.show()


### 그래프 3: 각 지표별 점수 분포 박스플롯

In [ ]:
# 그래프 3: 각 지표별 점수 분포 박스플롯
fig, axes = plt.subplots(1, 3, figsize=(18, 5))

metrics = ['relevancy_scores', 'faithfulness_scores', 'geval_scores']
titles = ['Answer Relevancy', 'Faithfulness', 'G-Eval']
colors = ['#FF6B6B', '#4ECDC4']

for idx, (metric, title, ax) in enumerate(zip(metrics, titles, axes)):
    data = [base_results[metric], finetuned_results[metric]]
    
    bp = ax.boxplot(data, labels=['Base Model', 'Fine-tuned Model'],
                    patch_artist=True, widths=0.6)
    
    # 박스 색상 설정
    for patch, color in zip(bp['boxes'], colors):
        patch.set_facecolor(color)
        patch.set_alpha(0.7)
    
    ax.set_title(title, fontsize=13, fontweight='bold', pad=10)
    ax.set_ylabel('점수', fontsize=11)
    ax.set_ylim(0, 1.0)
    ax.grid(axis='y', alpha=0.3, linestyle='--')
    
    # 평균값 표시
    means = [np.mean(d) for d in data]
    ax.plot([1, 2], means, 'D-', color='red', markersize=8, linewidth=2, 
            label='평균', alpha=0.7)
    
    if idx == 0:
        ax.legend(loc='lower right', fontsize=9)

plt.suptitle('평가 지표별 점수 분포', fontsize=15, fontweight='bold', y=1.02)
plt.tight_layout()
plt.show()


### 그래프 4: 레이더 차트 (종합 성능 비교)

In [ ]:
# 그래프 4: 레이더 차트 (종합 성능 비교)
from matplotlib.patches import Circle, RegularPolygon
from matplotlib.path import Path
from matplotlib.projections.polar import PolarAxes
from matplotlib.projections import register_projection
from matplotlib.spines import Spine
from matplotlib.transforms import Affine2D

def radar_factory(num_vars, frame='circle'):
    """레이더 차트를 위한 극좌표 축 생성"""
    theta = np.linspace(0, 2*np.pi, num_vars, endpoint=False)
    
    class RadarAxes(PolarAxes):
        name = 'radar'
        
        def __init__(self, *args, **kwargs):
            super().__init__(*args, **kwargs)
            self.set_theta_zero_location('N')
        
        def fill(self, *args, closed=True, **kwargs):
            return super().fill(closed=closed, *args, **kwargs)
        
        def plot(self, *args, **kwargs):
            lines = super().plot(*args, **kwargs)
            for line in lines:
                self._close_line(line)
        
        def _close_line(self, line):
            x, y = line.get_data()
            if x[0] != x[-1]:
                x = np.concatenate((x, [x[0]]))
                y = np.concatenate((y, [y[0]]))
                line.set_data(x, y)
        
        def set_varlabels(self, labels):
            self.set_thetagrids(np.degrees(theta), labels)
        
        def _gen_axes_patch(self):
            return Circle((0.5, 0.5), 0.5)
        
        def _gen_axes_spines(self):
            spine = Spine(axes=self,
                         spine_type='circle',
                         path=Path.unit_circle())
            spine.set_transform(Affine2D().scale(.5).translate(.5, .5)
                              + self.transAxes)
            return {'polar': spine}
    
    register_projection(RadarAxes)
    return theta




In [ ]:
# 레이더 차트 생성
fig = plt.figure(figsize=(10, 10))
theta = radar_factory(3, frame='circle')

ax = fig.add_subplot(111, projection='radar')
ax.set_rgrids([0.2, 0.4, 0.6, 0.8, 1.0])

# 데이터 준비
categories = ['Answer\nRelevancy', 'Faithfulness', 'G-Eval']
base_values = [base_avg['answer_relevancy'], base_avg['faithfulness'], base_avg['geval']]
finetuned_values = [finetuned_avg['answer_relevancy'], finetuned_avg['faithfulness'], finetuned_avg['geval']]

# 차트 그리기
ax.plot(theta, base_values, 'o-', linewidth=2, label='Base Model', color='#FF6B6B')
ax.fill(theta, base_values, alpha=0.25, color='#FF6B6B')
ax.plot(theta, finetuned_values, 'o-', linewidth=2, label='Fine-tuned Model', color='#4ECDC4')
ax.fill(theta, finetuned_values, alpha=0.25, color='#4ECDC4')

ax.set_varlabels(categories)
ax.set_ylim(0, 1.0)
ax.legend(loc='upper right', bbox_to_anchor=(1.3, 1.1), fontsize=11)
plt.title('종합 성능 레이더 차트', size=14, fontweight='bold', pad=20)

plt.tight_layout()
plt.show()


## 8. 평가 결과 분석

In [ ]:
# 상세 분석 정보 출력
print("="*80)
print("상세 분석 리포트")
print("="*80)

for metric_name in ['Answer Relevancy', 'Faithfulness', 'G-Eval']:
    row = results_df[results_df['Metric'] == metric_name].iloc[0]
    
    print(f"\n{'='*80}")
    print(f"{metric_name}")
    print(f"{'='*80}")
    print(f"  Base Model 평균: {row['Base Model']:.4f}")
    print(f"  Fine-tuned Model 평균: {row['Fine-tuned Model']:.4f}")
    print(f"  성능 향상률: {row['Improvement']:+.2f}%")
    
    # 통계 정보
    if metric_name == 'Answer Relevancy':
        base_scores = base_results['relevancy_scores']
        ft_scores = finetuned_results['relevancy_scores']
    elif metric_name == 'Faithfulness':
        base_scores = base_results['faithfulness_scores']
        ft_scores = finetuned_results['faithfulness_scores']
    else:
        base_scores = base_results['geval_scores']
        ft_scores = finetuned_results['geval_scores']
    
    print(f"\n  Base Model 통계:")
    print(f"    - 최소값: {np.min(base_scores):.4f}")
    print(f"    - 최대값: {np.max(base_scores):.4f}")
    print(f"    - 표준편차: {np.std(base_scores):.4f}")
    
    print(f"\n  Fine-tuned Model 통계:")
    print(f"    - 최소값: {np.min(ft_scores):.4f}")
    print(f"    - 최대값: {np.max(ft_scores):.4f}")
    print(f"    - 표준편차: {np.std(ft_scores):.4f}")
